This notebook generates all plots regarding the use of the immune dictionary data as a test dataset for our IL-2 model to correlate model-simulated pSTAT5 to overall changes in the transcriptome

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import use
from scipy.stats import spearmanr,pearsonr
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from collections import Counter

In [2]:
# Load simualtions and the iummune dictionary data on overall transcriptomic changes
df_sim = pd.read_csv("results/immune_dict/simulations_cytk_dict.csv")
df_cytk_dict = pd.read_excel("data/signaling/41586_2023_6816_MOESM13_ESM.xlsx", sheet_name='Fig1d')
df_cytk_dict = df_cytk_dict.loc[df_cytk_dict["Cytokine"]=="IL-2"][["Celltype", "Magnitude", "Magnitude_norm"]]
df_cytk_dict = df_cytk_dict.loc[df_cytk_dict["Celltype"] != "Langerhans"]
df_cytk_dict = df_cytk_dict.loc[df_cytk_dict["Celltype"] != "Mast cell"]
df_cytk_dict["Celltype"] = ["B_cell", "T_cell_CD4", "T_cell_CD8", "T_cell_gd", "Treg", "NK_cell", "ILC", "pDC", "cDC1", "cDC2", "MigDC", "eTAC", "Macrophage", "Monocyte", "Neutrophil"]
df_sim = df_sim[["Cell_type","RA0","RB0","RG0","STAT50","Result"]]
df_sim.columns = ["Celltype","RA0","RB0","RG0","STAT50","Result"]
df_sim["Celltype"] = [cell.split("__")[0] for cell in df_sim["Celltype"]]
df = pd.merge(df_cytk_dict, df_sim, on='Celltype')
df.loc[df["Magnitude_norm"] == '≥100',"Magnitude_norm"] = str(100)
df["Magnitude_norm"] = df["Magnitude_norm"].astype("double")
df["Magnitude"] = df["Magnitude"].astype("double")
df["Result_log10"] = np.log10(df["Result"]/df["Result"].max()*100)

In [3]:
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
sns.regplot(x="Result_log10", y="Magnitude_norm", data=df, ci=95, scatter_kws={'zorder': 10, 's': 100, 'color': 'xkcd:blue'}, line_kws={'color': "darkred", 'linewidth': 5})
plt.ylim(0, 105)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.ylabel('mRNA response (%)', fontsize=25)
plt.xlabel('log(pSTAT5) (%)', fontsize=25)
plt.xticks([0,0.5,1,1.5,2], [0,5,10,50,100])
plt.text(-0.25, 100, '$r_s=$'+str(round(spearmanr(df["Result"], df["Magnitude_norm"])[0], 2))+", $p=$"+str(round(spearmanr(df["Result"], df["Magnitude_norm"])[1], 3))+'\n$ρ=$'+str(round(pearsonr(df["Result"], df["Magnitude_norm"])[0], 2))+", $p=$"+str(round(pearsonr(df["Result"], df["Magnitude_norm"])[1], 3)), ha='left', va='top', fontsize=20, color='black', bbox=dict(facecolor='white', edgecolor='grey', boxstyle='round', linewidth=4))
plt.savefig("figures/cytokine_dict/Final_correlation_cytk_dict.pdf", transparent=True, bbox_inches="tight")
plt.close()